# NB148: The Complete Map — 48 CRT Elements ↔ 48 SM Fermions

**Goal**: Build the explicit bijection between the 48 elements of Z*₂₁₀ and the 48 Standard Model fermion states, combining all established tools:
- NB62: Level 1 Color Theorem (3+1 split from |Im₁|), chirality gap (a₃), charge sector (a₅), generation (a₇ mod 3)
- NB140: Wreath product decomposition (6 = 3+1+1+1 and 4 = 2+1+1)
- NB145-146: Three-layer mechanism (generator mod-p profile selects realization)
- NB69-78: CP pair mass predictions

**Strategy**: 
1. Start from NB62's established assignments (which are VERIFIED against the Cayley Laplacian)
2. For each of the 16 per-generation types: identify the SM fermion by its gauge quantum numbers
3. Extend to all 3 generations
4. Verify the resulting map is consistent with CP pair mass predictions
5. Check where the map is determined (forced by the algebra) vs where it requires dynamics

## S0: The NB62 Dictionary — What Is Already Established

NB62 established the following assignment of CRT quantum numbers to SM properties:

| CRT label | Group | SM property | How determined |
|-----------|-------|-------------|----------------|
| a₃ ∈ {0, 1} | Z₂ from φ(3) | **Chirality** (L/R) | ΔE(a₃) = 6, uniform across all sectors. a₃=0 lighter. |
| a₅ ∈ {0, 1, 2, 3} | Z₄ from φ(5) | **Charge sector** | a₅=0: constructive (down+lep, 3+1 color). a₅=1,3: isospin doublet (Im₂ cancellation). a₅=2: tower-protected (|S|=0). |
| a₇ mod 3 | Z₃ ⊂ Z₆ from φ(7) | **Generation** | Gen0={0,3}, Gen1={1,4}, Gen2={2,5}. Convention: Gen0=3rd (heaviest), Gen2=1st (lightest). |
| (a₃, a₇) within a₅=0 | Z₂ × Z₆ | **Color vs lepton** | |Im₁| = 3√3/2 → lepton (1 state); |Im₁| = √3/2 → quark (3 states). The 3+1 split. |

What NB62 did NOT establish:
- The specific SM identity of each a₅=1 and a₅=3 state (up-type vs down-type within the doublet)
- The specific color label (r, g, b) for each quark state
- The connection between the a₅=2 protected sector and neutrinos

These are the gaps this notebook will investigate.

In [3]:
# ── S0: Reproduce and extend NB62's complete table ──

import sys, os, numpy as np
from pathlib import Path
from math import gcd
from collections import defaultdict

ROOT = Path.cwd().parent
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

from solenoid_algebra import SA

print('S0: THE NB62 DICTIONARY — COMPLETE TABLE')
print('='*70)

# Reproduce all eigenvalue data for the 48 characters
coupled_gens = [17, 23, 37]
all_indices = SA.all_character_indices()  # list of (a2, a3, a5, a7) tuples

# Characters organized by CRT
# inter_pairs are pairs (i, j) of indices into all_indices such that
# all_indices[i] and all_indices[j] are in different generations
# But we need ALL 48, not just inter-gen pairs

# Compute Im_1, Im_2, and tower S for each of the 48 characters
ACTIVE_PRIMES = [[3], [3, 7], [3, 5, 7]]

def chi_val_level(idx, k, level):
    """Character value at element k, at covering tower level."""
    active = ACTIVE_PRIMES[level]
    phase = 0.0
    primes_list = [2, 3, 5, 7]
    phis = [1, 2, 4, 6]
    for p, phi_p, a in zip(primes_list, phis, idx):
        if phi_p == 1:
            continue
        if p not in active:
            continue
        r = k % p
        if r in SA.dlog[p]:
            phase += 2 * np.pi * a * SA.dlog[p][r] / phi_p
    return np.exp(1j * phase)

# Build the complete 48-element table
print(f'{"Gen":>3s} {"a3":>3s} {"a5":>3s} {"a7":>3s} | {"Im1":>8s} {"Im2":>8s} {"S":>8s} | '
      f'{"Sector":>10s} {"Color":>8s} {"#106":>5s}')
print('-' * 80)

table = []
for idx in all_indices:
    a2, a3, a5, a7 = idx
    gen = a7 % 3
    
    # Compute Im at each level
    im_levels = []
    for lev in range(3):
        im = sum(chi_val_level(idx, g, lev).imag for g in coupled_gens)
        im_levels.append(im)
    
    im1 = im_levels[0]  # Level 0: active primes {3}... wait
    # NB62 convention: Level 1 uses {3,7}, Level 2 uses {3,5,7}
    # im_levels[0] = active {3} only
    # im_levels[1] = active {3,7} = "Level 1" in NB62
    # im_levels[2] = active {3,5,7} = "Level 2" in NB62
    
    im1_nb62 = im_levels[1]  # NB62's "Im_1" uses {3,7}
    im2_nb62 = im_levels[2] - im_levels[1]  # NB62's "Im_2" is the increment from adding p=5
    s_tower = im_levels[2]  # Total tower S
    
    # Sector assignment from NB62
    sector_name = {0: 'down+lep', 1: 'mixed_A', 2: 'protected', 3: 'mixed_B'}[a5]
    
    # Color assignment (only for a5=0 in Gen1/Gen2)
    abs_im1 = abs(im1_nb62)
    s3 = np.sqrt(3)
    if a5 == 0 and gen in [1, 2]:
        if abs(abs_im1 - 3*s3/2) < 0.01:
            color = 'LEPTON'
        elif abs(abs_im1 - s3/2) < 0.01:
            color = 'quark'
        else:
            color = '???'
    else:
        color = ''
    
    # Identity #106 check
    check_106 = ''
    if a5 == 0 and gen in [1, 2]:
        check_106 = 'L' if color == 'LEPTON' else 'q'
    
    row = {
        'a2': a2, 'a3': a3, 'a5': a5, 'a7': a7, 'gen': gen,
        'im1': im1_nb62, 'im2': im2_nb62, 's_tower': s_tower,
        'sector': sector_name, 'color': color,
        'abs_im1': abs_im1
    }
    table.append(row)
    
    if gen == 1:  # Show Gen1 only (Gen2 is conjugate, Gen0 has no inter-gen data)
        print(f'{gen:3d} {a3:3d} {a5:3d} {a7:3d} | {im1_nb62:+8.4f} {im2_nb62:+8.4f} {s_tower:+8.4f} | '
              f'{sector_name:>10s} {color:>8s} {check_106:>5s}')

# Summary per a5 sector
print(f'\n\nPER-SECTOR SUMMARY (Gen1):')
for a5 in range(4):
    sector_rows = [r for r in table if r['a5'] == a5 and r['gen'] == 1]
    n_quark = sum(1 for r in sector_rows if r['color'] == 'quark')
    n_lepton = sum(1 for r in sector_rows if r['color'] == 'LEPTON')
    n_unassigned = len(sector_rows) - n_quark - n_lepton
    sector = {0: 'down+lep', 1: 'mixed_A', 2: 'protected', 3: 'mixed_B'}[a5]
    print(f'  a₅={a5} ({sector}): {len(sector_rows)} states, '
          f'{n_quark} quark, {n_lepton} lepton, {n_unassigned} unassigned')

# The SM fermion types per generation (16 total):
# We need to map these 16 CRT states to:
# (3,2): Q_L = u_L + d_L (6 states: 3 colors × 2 isospin)
# (3,1): u_R (3 states: 3 colors)
# (3,1): d_R (3 states: 3 colors)
# (1,2): L_L = ν_L + e_L (2 states)
# (1,1): e_R (1 state)
# (1,1): ν_R (1 state)
# Total: 6 + 3 + 3 + 2 + 1 + 1 = 16

print(f'\n\nSM FERMION TYPES PER GENERATION:')
print(f'  SU(3) triplet (quarks):  3 colors × 4 types (u_L, d_L, u_R, d_R) = 12')
print(f'  SU(3) singlet (leptons): 1 × 4 types (ν_L, e_L, e_R, ν_R) = 4')
print(f'  Total: 12 + 4 = 16 = d(210)')

# From NB62: a5=0 has 3+1 split (3 quarks + 1 lepton).
# The 4 states in a5=0 (for Gen1): 
#   (a3=0, a7=1): LEPTON
#   (a3=0, a7=4): quark
#   (a3=1, a7=1): quark
#   (a3=1, a7=4): quark

# Each of the other sectors (a5=1,2,3) also has 4 states.
# Total: 4 × 4 = 16 per generation.

# The key question: how do the 12 quark + 4 lepton states
# distribute across the 4 sectors × (2 a3 × 2 a7) = 16 CRT slots?

# From a5=0: 3 quarks + 1 lepton = 4 ✓
# From a5=1: 4 states (mixed_A)
# From a5=2: 4 states (protected)
# From a5=3: 4 states (mixed_B)

# The 12 remaining states (from a5=1,2,3) must contain 9 quarks + 3 leptons.
# But NB62 doesn't assign color/lepton to a5≠0 sectors!
# The |Im_1| 3+1 split only works within a5=0.

print(f'\n\nTHE ASSIGNMENT PROBLEM:')
print(f'  a5=0: 3 quarks + 1 lepton = 4 (from NB62 #106)')
print(f'  a5=1,2,3: 12 states unassigned → must be 9 quarks + 3 leptons')
print(f'  OR: the a5 sectors don\'t simply partition into quarks and leptons.')
print(f'  The SM has quarks across ALL charge sectors (u, d have different charges).')
print(f'  The a5 label is CHARGE TYPE, not quark/lepton identity.')

S0: THE NB62 DICTIONARY — COMPLETE TABLE
Gen  a3  a5  a7 |      Im1      Im2        S |     Sector    Color  #106
--------------------------------------------------------------------------------
  1   0   0   1 |  +2.5981  +0.0000  +2.5981 |   down+lep   LEPTON     L
  1   0   0   4 |  +0.8660  +0.0000  +0.8660 |   down+lep    quark     q
  1   0   1   1 |  +2.5981  -2.0981  +0.5000 |    mixed_A               
  1   0   1   4 |  +0.8660  -1.3660  -0.5000 |    mixed_A               
  1   0   2   1 |  +2.5981  -5.1962  -2.5981 |  protected               
  1   0   2   4 |  +0.8660  -1.7321  -0.8660 |  protected               
  1   0   3   1 |  +2.5981  -3.0981  -0.5000 |    mixed_B               
  1   0   3   4 |  +0.8660  -0.3660  +0.5000 |    mixed_B               
  1   1   0   1 |  -0.8660  +0.0000  -0.8660 |   down+lep    quark     q
  1   1   0   4 |  +0.8660  +0.0000  +0.8660 |   down+lep    quark     q
  1   1   1   1 |  -0.8660  -0.6340  -1.5000 |    mixed_A               
  

## S1: The Key Realization — Im₁ Is a₅-Independent

NB62 cell 3 explicitly states: "PASS: Im_1 is independent of a5 (p=5 invisible at level 1)." This means the 3+1 color-lepton split from |Im₁| applies to ALL a₅ sectors, not just a₅=0. The same (a₃, a₇) combination that is a lepton at a₅=0 is ALSO a lepton at a₅=1, a₅=2, and a₅=3.

This changes the assignment completely. Each a₅ sector has 3 quarks + 1 lepton (from the same (a₃,a₇) split). So per generation:
- 4 a₅ sectors × 3 quarks each = 12 quark states
- 4 a₅ sectors × 1 lepton each = 4 lepton states
- Total: 12 + 4 = 16 ✓

The a₅ label then distinguishes different CHARGE TYPES within the quark and lepton sectors:
- a₅=0: "down+lep" sector (constructive, largest inter-gen split)
- a₅=1: "mixed_A" (isospin partner A)
- a₅=2: "protected" (zero inter-gen split)
- a₅=3: "mixed_B" (isospin partner B)

Each quark comes in 4 charge types × 3 colors × 2 chiralities... that's 24, not 12. So the CRT must be organizing this differently. Let me compute carefully.

In [5]:
# ── S1: Im₁-based assignment across all a₅ sectors ──

print('S1: Im₁-BASED QUARK/LEPTON ASSIGNMENT — ALL SECTORS')
print('='*70)

# Im₁ depends only on (a₃, a₇), not a₅. So the 3+1 split applies everywhere.
# For Gen1: the lepton (a₃, a₇) is (0, 1). The quarks are (0,4), (1,1), (1,4).

# Map all 16 Gen1 states
print(f'Gen1 complete assignment (using Im₁ 3+1 across all a₅):')
print(f'{"a3":>3s} {"a5":>3s} {"a7":>3s} | {"type":>8s} {"sector":>10s} | {"SM candidate":>25s}')
print('-' * 65)

for a5 in range(4):
    for a3 in range(2):
        for a7 in [1, 4]:
            # Im₁-based color/lepton: (0,1) = lepton, rest = quark
            is_lepton = (a3 == 0 and a7 == 1)
            particle_type = 'LEPTON' if is_lepton else 'quark'
            sector = {0: 'down+lep', 1: 'mixed_A', 2: 'protected', 3: 'mixed_B'}[a5]
            
            # SM candidate based on:
            # - a₃: chirality (0=L, 1=R)
            # - a₅: charge type
            # - is_lepton: from 3+1 split
            chirality = 'L' if a3 == 0 else 'R'
            
            if is_lepton:
                # Lepton: which type depends on a₅
                if a5 == 0:
                    sm = f'e_{chirality}' if a3 == 0 else f'e_{chirality}'
                elif a5 == 2:
                    sm = f'ν_{chirality}'  # protected = neutrino?
                else:
                    sm = f'lep_{chirality}(a5={a5})'
            else:
                # Quark: which type depends on a₅ and (a₃, a₇)
                # 3 quarks per sector: need to assign colors
                sm = f'q_{chirality}(a5={a5})'
            
            print(f'{a3:3d} {a5:3d} {a7:3d} | {particle_type:>8s} {sector:>10s} | {sm:>25s}')
    print()

# Count
n_quarks = sum(1 for a5 in range(4) for a3 in range(2) for a7 in [1,4] 
               if not (a3==0 and a7==1))
n_leptons = sum(1 for a5 in range(4) for a3 in range(2) for a7 in [1,4] 
                if (a3==0 and a7==1))
print(f'Quarks: {n_quarks}, Leptons: {n_leptons}, Total: {n_quarks + n_leptons}')

# Wait: 12 quarks, 4 leptons. That's exactly right!
# But the 12 quarks are: 4 a₅ × 3 (a₃,a₇) combos.
# In the SM: 12 quarks = 3 colors × 4 types (u_L, d_L, u_R, d_R)

# The mapping would be:
# The 3 quark (a₃,a₇) values → 3 colors
# The 4 a₅ values → 4 quark types
# But 4 quark types per color = u_L, d_L, u_R, d_R → needs chirality (a₃)
# Chirality is ALREADY in a₃, but a₃ is also part of the color label!

# The issue: a₃ does DOUBLE DUTY:
# For LEPTONS: a₃ determines chirality (L vs R)
# For QUARKS: a₃ is PART OF the color assignment (one of the 3 (a₃,a₇) combos)

# But the SM's quarks ALSO have chirality! u_L and u_R are different states.
# So we can't use a₃ for both color and chirality simultaneously.

print(f'\n\nTHE DOUBLE-DUTY PROBLEM:')
print(f'  The 3 quark (a₃,a₇) values are: (0,4), (1,1), (1,4)')
print(f'  These determine COLOR (3 values).')
print(f'  But a₃ also determines CHIRALITY (L/R).')
print(f'  The quark (0,4) has a₃=0 (left-handed?)')
print(f'  The quarks (1,1) and (1,4) have a₃=1 (right-handed?)')
print(f'  But the SM has both left AND right quarks of EACH color.')
print()
print(f'  RESOLUTION POSSIBILITIES:')
print(f'  A) The 3 (a₃,a₇) combos are NOT colors. They\'re something else.')
print(f'  B) Chirality for quarks comes from a₅ (charge sector), not a₃.')
print(f'  C) The CRT does not map one-to-one to (color, chirality, type).')
print(f'     Instead, the 16 CRT states are 16 MODES that mix to form')
print(f'     physical states through the cascade dynamics.')
print()
print(f'  NB62 named the a₃=0 sector "lighter" and a₃=1 "heavier"')
print(f'  (Delta_E = 6). But the 3+1 split mixes a₃ values in the')
print(f'  quark triplet: 1 has a₃=0, 2 have a₃=1. This means the')
print(f'  color triplet is NOT aligned with chirality — it crosses')
print(f'  the chirality boundary.')

S1: Im₁-BASED QUARK/LEPTON ASSIGNMENT — ALL SECTORS
Gen1 complete assignment (using Im₁ 3+1 across all a₅):
 a3  a5  a7 |     type     sector |              SM candidate
-----------------------------------------------------------------
  0   0   1 |   LEPTON   down+lep |                       e_L
  0   0   4 |    quark   down+lep |                 q_L(a5=0)
  1   0   1 |    quark   down+lep |                 q_R(a5=0)
  1   0   4 |    quark   down+lep |                 q_R(a5=0)

  0   1   1 |   LEPTON    mixed_A |               lep_L(a5=1)
  0   1   4 |    quark    mixed_A |                 q_L(a5=1)
  1   1   1 |    quark    mixed_A |                 q_R(a5=1)
  1   1   4 |    quark    mixed_A |                 q_R(a5=1)

  0   2   1 |   LEPTON  protected |                       ν_L
  0   2   4 |    quark  protected |                 q_L(a5=2)
  1   2   1 |    quark  protected |                 q_R(a5=2)
  1   2   4 |    quark  protected |                 q_R(a5=2)

  0   3   1 |   L

## S2: The Entanglement — Color and Chirality in the CRT Basis

The double-duty problem reveals that the CRT eigenstates are NOT the SM physical eigenstates for quarks. In the SM, color and chirality are independent quantum numbers: a red left-handed up quark (u_L^r) has definite color AND definite chirality. But in the CRT, the color triplet {(0,4), (1,1), (1,4)} has MIXED a₃ values — one state is "left" and two are "right."

This means the CRT eigenstates are eigenstates of the **Cayley Laplacian** (a character basis), while the SM physical states are eigenstates of the **gauge Hamiltonian** (a representation basis). The two bases are related by a change of basis — and understanding this change of basis IS the fermion bijection.

The wreath product analysis (NB140) already hints at this: the 3-dim irrep of Z₂ ≀ Z₃ operates in a basis where all three states are equivalent under the Z₃ cycling — they don't have definite a₃. The physical color states are the wreath product irrep basis vectors, not the CRT character basis vectors.

**The key question**: Can we identify the change-of-basis matrix from CRT characters to wreath product irreps? If yes, the physical fermion states are the transformed CRT modes, and the bijection is EXACT.

In [7]:
# ── S2: The change of basis from CRT characters to wreath product irreps ──

print('S2: CRT CHARACTER BASIS vs WREATH PRODUCT IRREP BASIS')
print('='*70)

# The CRT basis: 4 states per a₅ sector, labeled (a₃, a₇) for a₇ in gen1 {1,4}
# Im₁ values: (0,1)=+3√3/2, (0,4)=+√3/2, (1,1)=-√3/2, (1,4)=+√3/2

# The wreath product basis (from NB140 S8):
# 6D permutation rep on {(i,b): i∈{0,1,2}, b∈{0,1}} decomposes as
# Even parity (|i,+⟩): 3 states → 1+1+1 (generation singlets)
# Odd parity (|i,−⟩): 3 states → 3 (color triplet)

# The CRT has 4 states per (a₅, gen) sector, NOT 6.
# The 6 sheets correspond to (a₃, a₇) combinations across ALL generations.
# Within one generation: 2 (a₃) × 2 (a₇ values) = 4 states, not 6.

# Wait: I need to be more careful about which space the wreath product acts on.
# Z₂ ≀ Z₃ acts on 6 sheets = 3 positions × 2 bits.
# The 6 sheets of the 2×3 covering are:
# (i, b) with i ∈ {0,1,2} and b ∈ {0,1}
# In the CRT language:
#   i → related to a₇ mod 3 (the Z₃ part)
#   b → related to a₃ (the Z₂ part, bilateral)

# But within one generation (fixed a₇ mod 3), we have only 2 a₇ values 
# (e.g., {1,4} for gen1) and 2 a₃ values → 4 states, not 6.

# The 6 sheets span ACROSS generations:
# i=0 → gen0 (a₇ mod 3 = 0)
# i=1 → gen1 (a₇ mod 3 = 1)
# i=2 → gen2 (a₇ mod 3 = 2)
# b=0 → one a₇ value within the generation (e.g., a₇=0,1,2)
# b=1 → the other a₇ value (e.g., a₇=3,4,5)

# So the 6 (a₃-fixed, a₅-fixed) states across 3 generations are:
# (gen0, a₇=0), (gen0, a₇=3), (gen1, a₇=1), (gen1, a₇=4), (gen2, a₇=2), (gen2, a₇=5)
# This is the full set of 6 a₇ values!

# The wreath product acts on these 6 a₇ values (at fixed a₃ and a₅).
# The 3+1+1+1 decomposition then says:
# 3 of the 6 a₇ values form a triplet (transform together = color)
# 3 form singlets (independent = generation labels)

# But the 3+1 split from NB62 (|Im₁|) works WITHIN a single generation,
# splitting the 4 (a₃, a₇) states into 3 quarks + 1 lepton.
# The wreath product works ACROSS generations, splitting the 6 a₇ values.

# These are DIFFERENT decompositions acting on DIFFERENT spaces!

print(f'KEY CLARIFICATION:')
print(f'')
print(f'  The NB62 3+1 split operates WITHIN a generation:')
print(f'    4 states (2 a₃ × 2 a₇) → 3 quarks + 1 lepton')
print(f'    This is the quark/lepton distinction.')
print(f'')
print(f'  The NB140 3+1+1+1 split operates ACROSS generations:')
print(f'    6 a₇ values → 3 (triplet) + 1+1+1 (singlets)')
print(f'    This is the color/generation distinction.')
print(f'')
print(f'  They are NOT the same decomposition.')
print(f'  They act on different vector spaces.')
print(f'  The NB62 split uses (a₃, a₇) within a fixed a₅ and gen.')
print(f'  The NB140 split uses a₇ values across all generations at fixed a₃ and a₅.')
print(f'')

# Let me verify: are the 3 quark states from the NB62 split the SAME
# as the 3 states in the NB140 triplet?

# NB62 quarks (gen1, a₅=0): (a₃, a₇) = (0,4), (1,1), (1,4)
# These have a₇ values {4, 1, 4} → a₇ mod 3 = {1, 1, 1} → all gen1!
# So the NB62 "color" triplet is within ONE generation.

# NB140 triplet: 3 of the 6 a₇ values {0,1,2,3,4,5}
# These are ACROSS generations: e.g., a₇ = 0 (gen0), 1 (gen1), 2 (gen2)

# These are completely different things:
# - NB62 "color": 3 states within one generation, different (a₃,a₇) combos
# - NB140 "color": 3 generations of the same (a₃,a₅) type

print(f'CRITICAL FINDING:')
print(f'  NB62 "3 quarks" = 3 different (a₃,a₇) within gen1 = 3 "colors"')
print(f'    These are: (0,4), (1,1), (1,4) — all have a₇ mod 3 = 1 (gen1)')
print(f'')
print(f'  NB140 "triplet" = 3 generations of the same type')
print(f'    These span across gen0, gen1, gen2')
print(f'')
print(f'  They are different meanings of "3":')
print(f'    NB62: 3 = color (within-generation, from eigenvalue degeneracy)')
print(f'    NB140: 3 = generation-triplet (across-generation, from wreath irrep)')
print(f'')
print(f'  The NB140 wreath product does NOT directly produce color.')
print(f'  It produces the 3-fold generation structure.')
print(f'  Color comes from NB62\'s |Im₁| degeneracy WITHIN a generation.')
print(f'')
print(f'  THIS MEANS: the wreath product 6=3+1+1+1 is about')
print(f'  GENERATION STRUCTURE, not about color directly.')
print(f'  The "triplet" IS the generation triplet.')
print(f'  The "singlets" are a different structure entirely.')

# Wait, but NB140 identified the triplet as COLOR and singlets as generations.
# Let me reconsider...

print(f'\n\nRECONSIDERATION:')
print(f'  NB140 said: triplet = color (quarks), singlets = generations (leptons).')
print(f'  But the 6 sheets (i,b) with i ∈ {{0,1,2}} represent 3 POSITIONS')
print(f'  in the Z₃ covering, not 3 generations.')
print(f'  The Z₃ cycling permutes these positions — that\'s the "color rotation."')
print(f'  The 3+1+1+1 decomposition says: 3 positions transform together')
print(f'  (= color triplet), while the 3 singlets are the Z₃ Fourier modes')
print(f'  (= independent generation copies).')
print(f'')
print(f'  But these Z₃ positions are NOT the same as a₇ mod 3 generations.')
print(f'  They are the fiber positions in the 3-fold covering at level 2.')
print(f'  The connection to a₇ is through the covering constraint.')
print(f'')
print(f'  The NB62 within-generation "3 colors" come from the LEVEL 1')
print(f'  eigenvalue degeneracy — 3 (a₃,a₇) combos with same |Im₁|.')
print(f'  This is related to but not identical with the wreath product triplet.')
print(f'')
print(f'  HONEST STATUS: The relationship between NB62\'s within-generation')
print(f'  color degeneracy and NB140\'s across-generation wreath triplet')
print(f'  is not yet fully understood. Both give a "3", but they operate')
print(f'  on different spaces. Making this connection explicit is the')
print(f'  remaining challenge for the complete fermion map.')

S2: CRT CHARACTER BASIS vs WREATH PRODUCT IRREP BASIS
KEY CLARIFICATION:

  The NB62 3+1 split operates WITHIN a generation:
    4 states (2 a₃ × 2 a₇) → 3 quarks + 1 lepton
    This is the quark/lepton distinction.

  The NB140 3+1+1+1 split operates ACROSS generations:
    6 a₇ values → 3 (triplet) + 1+1+1 (singlets)
    This is the color/generation distinction.

  They are NOT the same decomposition.
  They act on different vector spaces.
  The NB62 split uses (a₃, a₇) within a fixed a₅ and gen.
  The NB140 split uses a₇ values across all generations at fixed a₃ and a₅.

CRITICAL FINDING:
  NB62 "3 quarks" = 3 different (a₃,a₇) within gen1 = 3 "colors"
    These are: (0,4), (1,1), (1,4) — all have a₇ mod 3 = 1 (gen1)

  NB140 "triplet" = 3 generations of the same type
    These span across gen0, gen1, gen2

  They are different meanings of "3":
    NB62: 3 = color (within-generation, from eigenvalue degeneracy)
    NB140: 3 = generation-triplet (across-generation, from wreath irrep)